In [1]:
%pip install python-dotenv openai datasets math_verify torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import os
from openai import AsyncOpenAI
from math_verify import parse

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

client = AsyncOpenAI(
	base_url="https://integrate.api.nvidia.com/v1",
	api_key=NVIDIA_API_KEY,
)

prompt = "What is 13 times 17? Box your answer."
gold = "221"

completion = await client.chat.completions.create(
	model="nvidia/nemotron-3.5-lightning-30b-a3b",
	messages=[{"role": "user", "content": prompt}],
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10
    },
)

print("*"*20, "Prompt", "*"*20)
print(prompt)

parsed_gold = parse(gold) or [None]
for i, choice in enumerate(completion.choices):
  message = choice.message
  parsed_answer = parse(message.content) or [None]
  correct = parsed_gold[0] == parsed_answer[0]

  print("*"*20, f"Generation {i+1}: {parsed_answer[0]} ({'correct' if correct else 'incorrect'})", "*"*20)
  if correct:
    print(message.reasoning_content)

******************** Prompt ********************
What is 13 times 17? Box your answer.
******************** Generation 1: 221 (correct) ********************
Here's a thinking process:

1.  **Identify the User's Request**: The user wants to know the product of 13 and 17, and they want the answer boxed.

2.  **Calculate 13 × 17**:
   - I can compute this directly: 13 × 17
   - Break it down: 13 × 17 = 13 × (10 + 7) = 130 + 91 = 221
   - Alternatively: 17 × 13 = (20 - 3) × 13 = 260 - 39 = 221
   - Or standard multiplication: 
        17
      × 13
      ----
        51  (17 × 3)
       170  (17 × 10)
      ----
      221

3.  **Format the Output**: The user said "Box your answer." I need to present the final answer in a box. Typically, this means using LaTeX \boxed{} or just plain text with brackets/boxes. I'll use \boxed{221} as is common in math contexts.

4.  **Construct the Response**: 
   - State the calculation briefly or just give the boxed answer.
   - Make sure it's clear and dir

In [4]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", "default")
ds

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'],
        num_rows: 93733
    })
})

In [15]:
import asyncio
from math_verify import verify
import torch
from datasets import Dataset

async def generate_dataset(prompts, golds, **kwargs):
	dataset_dict = {
		"prompt": [],
		"outputs": [],
		"advanatages": [],
	}
	futures = []
	for prompt in prompts:
		futures.append(client.chat.completions.create(
			**kwargs,
			messages=[{"role": "user", "content": prompt}],
		))
	completions = await asyncio.gather(*futures)
	for prompt, completion, gold in zip(prompts, completions, golds):
		gold = parse(gold)

		outputs = []
		rewards = []
		for choice in completion.choices:
			message = choice.message
			answer = parse(message.content)
			correct = verify(gold, answer)

			outputs.append(f"<think>{message.reasoning_content}</think>{message.content}")
			rewards.append(1.0 if correct else 0.0)
		rewards = torch.tensor(rewards)

		rewards_std = rewards.std()
		if rewards_std < 1e-5:
			advantages = torch.zeros_like(rewards)
		else:
			advantages = (rewards - rewards.mean()) / rewards_std

		dataset_dict["prompt"].append(prompt)
		dataset_dict["outputs"].append(outputs)
		dataset_dict["advanatages"].append(advantages.tolist())
	return Dataset.from_dict(dataset_dict)

generated_ds = await generate_dataset(
	prompts=[
		"Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
		"What is 8 times 3? Box your answer.",
	],
	golds=["3", "24"],
	model="nvidia/nemotron-3.5-lightning-30b-a3b",
	max_tokens=2**10,
    n=8,
	extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "reasoning_budget": 2**10
    },
)
generated_ds[:]

{'prompt': ["Pick an random integer from 1 to 3. Don't pick 2. Box your answer.",
  'What is 8 times 3? Box your answer.'],
 'outputs': [['<think>Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "Pick an random integer from 1 to 3. Don\'t pick 2. Box your answer."\n   - Constraints: \n     - Pick a random integer from 1 to 3\n     - Must NOT pick 2\n     - Output must be boxed\n\n2.  **Identify the Core Task:**\n   - Random selection from {1, 2, 3}\n   - Exclude 2\n   - So valid choices are 1 or 3\n   - "Random" with the constraint means I should pick either 1 or 3, but since I\'m an AI, I need to simulate randomness or just pick one consistently with the constraint.\n   - The user said "Don\'t pick 2", so I must output either 1 or 3.\n   - I\'ll pick one randomly (or just choose one). Since I can\'t truly randomize, I\'ll pick one, say 1 or 3. Maybe I should pick fairly. I\'ll just choose 1 or 3. Let\'s say 3. Or I could note the constraint and pick one.\n  